# Индивидуальный проект - начало...

## Данные + первый индекс

Цель пары:

1. выбрать датасет, который подойдёт для дальнейшего проекта
2. проверить, что в нём есть нужные типы полей
3. выбрать **одно уникальное поле**
4. реализовать свою `Hash Table`
5. добавить `resize` и `rehash`
6. использовать таблицу как индекс вида `уникальный ключ → row_id`
7. сравнить поиск полным проходом и поиск через индекс

На следующих занятиях проект будет постепенно расширяться новыми структурами и алгоритмами, поэтому к выбору датасета нужно подойти ответственно


---
# 1. Требования к датасету

Выберите датасет, с которым будете работать дальше в течение этого семестра

### Обязательные требования

Датасет должен содержать:

- желательно **не менее 100 000 строк**
- минимум **5 содержательных столбцов**
- минимум **одно уникальное поле** - идентификатор записи
- минимум **одно числовое поле**
- минимум **одно категориальное поле**
- **два поля, которые описывают связь между объектами**, и ещё одно числовое поле, которое характеризует эту связь. Например:
  - аэропорт отправления → аэропорт назначения, число - длительность полёта / цена билета / расстояние
  - отправитель → получатель, число - сумма перевода
  - пользователь → товар, число - цена / количество / оценка
  - пользователь → трек, число - количество прослушиваний / оценка / длительность прослушивания
  - команда 1 → команда 2, число - разница счёта / количество очков / длительность матча
  - пользователь 1 → пользователь 2, число - количество сообщений / взаимодействий / совместных действий
- минимум **одно поле, по которому значения можно упорядочить**:
  - число
  - дата/время
  - рейтинг
  - длительность и т. п.

### Желательно

Хорошо, если есть дата/время - это пригодится в следующих задачах

### Пример подходящего датасета

Датасет покупок в маркетплейсе:

```text
purchase_id   - уникальный идентификатор покупки
user_id       - идентификатор пользователя
product_id    - идентификатор товара
quantity      - количество купленного товара
price         - цена товара
category      - категория товара
created_at    - дата и время покупки
```

Почему подходит:

- `purchase_id` - уникальное поле
- `price`, `quantity` - числовые поля
- `category` - категориальное поле
- `created_at` - поле, значения которого можно упорядочить
- `user_id → product_id` - связь между двумя объектами: пользователь купил товар
- `quantity` или `price` - числовая характеристика этой связи

Одну строку можно прочитать так:

> Пользователь `user_id` купил товар `product_id` в количестве `quantity` по цене `price` в момент `created_at`.

**Сейчас использовать связь между объектами в коде не нужно. Она понадобится на следующих занятиях**


## Коротко опишите свой датасет

**Название / источник:**  
Chicago Taxi Trips (2024-), портал открытых данных города Чикаго: https://data.cityofchicago.org/d/ajtu-isnz

**Что представляет одна строка:**  
Одна поездка на такси в Чикаго.

**Размер датасета:**  
200 000 строк, 23 столбца

**Какие поля есть:**  
trip_id, taxi_id, trip_start_timestamp, trip_end_timestamp, trip_seconds, trip_miles, pickup_census_tract, dropoff_census_tract, pickup_community_area, dropoff_community_area, fare, tips, tolls, extras, trip_total, payment_type, company, pickup_centroid_latitude, pickup_centroid_longitude, pickup_centroid_location, dropoff_centroid_latitude, dropoff_centroid_longitude, dropoff_centroid_location

**Какое поле является уникальным идентификатором:**  
trip_id

**Какое поле числовое:**  
trip_miles trip_seconds, fare, tips, trip_total

**Какое поле категориальное:**  
payment_type, company

**Какое поле можно использовать как упорядочиваемое:**  
trip_start_timestamp

**Какие два поля описывают связь между объектами:**  
pickup_community_area → dropoff_community_area

**Какое числовое поле характеризует эту связь:**  
trip_miles (также trip_total)

---
# 2. Загружаем данные

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from time import perf_counter

## Загрузка файла по ссылке

In [ ]:
DATA_PATH = "https://data.cityofchicago.org/resource/ajtu-isnz.csv?$limit=200000&$where=trip_start_timestamp%3E%3D%272025-01-01T00:00:00%27"

print("DATA_PATH =", DATA_PATH)

In [ ]:
def load_dataset(path):
    path = str(path)
    suffix = Path(path.split("?")[0]).suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    raise ValueError(f"Неизвестный формат: {suffix}")

df = load_dataset(DATA_PATH)

print("Размер:", df.shape)
display(df.head())

In [ ]:
print("Столбцы:")
for i, col in enumerate(df.columns):
    print(f"{i:>2}. {col:<30} dtype={df[col].dtype}")

print()

if len(df) >= 100_000:
    print("Размер датасета подходит")
else:
    print(f"В датасете {len(df):,} строк. Подумай над своими действиями...")

---
# 3. Выбираем уникальное поле

Сегодня строим индекс только для **уникального ключа**

Примеры:

```text
идентификатор покупки
идентификатор транзакции
идентификатор поездки
идентификатор рейса
```

Для такого поля будем хранить:

```text
уникальный ключ → row_id
```

Например:

```text
purchase_id 837291 → строка 41827
```


In [ ]:
UNIQUE_FIELD = "trip_id"

assert UNIQUE_FIELD in df.columns, (
    f"Столбца {UNIQUE_FIELD!r} нет. "
    f"Доступные столбцы: {list(df.columns)}"
)

series = df[UNIQUE_FIELD]

print("Поле:", UNIQUE_FIELD)
print("Количество строк:", len(series))
print("Уникальных значений:", series.nunique(dropna=False))
print("Пропущенных значений:", series.isna().sum())
print("Поле уникально:", series.is_unique)

### Требование

Для основной части работы поле должно быть:

- **уникальным**
- **без пропущенных значений**

Если поле не проходит проверку, выберите другое

Если в датасете нет естественного уникального идентификатора - обсудите это с преподавателем


In [ ]:
assert series.is_unique, (
    "Выбранное поле не уникально"
    "Для сегодняшнего задания выберите уникальный идентификатор"
)

assert not series.isna().any(), (
    "В уникальном поле есть пропущенные значения"
    "Выберите поле без пропусков"
)

keys = series.tolist()
print("Уникальный ключ выбран")


---
# 4. Поиск без индекса

Сначала реализуем обычный поиск полным проходом

Функция должна вернуть `row_id` найденной записи или `None`

In [ ]:
def linear_search(values, target):
    for row_id, value in enumerate(values):
        if value == target:
            return row_id
    return None

In [ ]:
TARGET = keys[len(keys) // 2]

row_id = linear_search(keys, TARGET)

print("TARGET:", TARGET)
print("row_id:", row_id)

if row_id is not None:
    display(df.iloc[[row_id]])

### Ответьте

**Временная сложность в худшем случае:**  
O(N), нужно пройти по всем ключам

**Дополнительная память:**  
O(1)

**Почему при миллионах записей такой поиск может быть проблемой:**  
Каждый запрос проходит по всем записям. Если запросов много, а записей миллионы, время растёт как число запросов * N и становится очень большим.

---
# 5. Реализуем Hash Table

Идея:

```text
key
 ↓
hash(key)
 ↓
hash(key) % capacity
 ↓
bucket
```

Разные ключи могут попасть в один `bucket` - это **коллизия**

Коллизии в этой работе решаем методом **separate chaining**:

```text
bucket → [(key1, value1), (key2, value2), ...]
```

## Обязательное расширение таблицы

Таблица начинается с небольшой вместимости:

```text
capacity = 8
```

Коэффициент заполнения:

```text
load_factor = size / capacity
```

В нашей реализации используем порог:

```text
MAX_LOAD_FACTOR = 0.75
```

Если добавление **нового** ключа приведёт к:

```text
(size + 1) / capacity > 0.75
```

таблица должна:

1. увеличить `capacity` в 2 раза
2. создать новые пустые `buckets`
3. заново распределить по ним **все старые пары**
4. после этого добавить новый элемент

Это называется **rehash**

Почему нельзя просто увеличить список `buckets`?

Потому что номер bucket вычисляется так:

```text
hash(key) % capacity
```

После изменения `capacity` тот же ключ может получить уже другой номер bucket

> `0.75` - порог, который мы используем в нашей учебной реализации. Это не универсальное правило для всех Hash Table

### Ограничение

Внутри `MyHashMap` нельзя использовать:

- `dict`
- `set`
- `defaultdict`


In [ ]:
class MyHashMap:
    MAX_LOAD_FACTOR = 0.75

    def __init__(self, capacity=8):
        self.capacity = capacity
        self.size = 0
        self.buckets = [[] for _ in range(capacity)]

    def _index(self, key):
        return hash(key) % self.capacity

    def load_factor(self):
        return self.size / self.capacity

    def _resize(self):
        old_buckets = self.buckets
        self.capacity *= 2
        self.buckets = [[] for _ in range(self.capacity)]

        for bucket in old_buckets:
            for key, value in bucket:
                self.buckets[self._index(key)].append((key, value))

    def put(self, key, value):
        bucket = self.buckets[self._index(key)]

        for i, (_key, _value) in enumerate(bucket):
            if _key == key:
                bucket[i] = (key, value)
                return
            
        if (self.size + 1) / self.capacity > self.MAX_LOAD_FACTOR:
            self._resize()
            bucket = self.buckets[self._index(key)]

        bucket.append((key, value))
        self.size += 1

    def get(self, key):
        index = self._index(key)
        for _key, value in self.buckets[index]:
            if key == _key:
                return value
        return None

    def contains(self, key):
        index = self._index(key)
        for _key, value in self.buckets[index]:
            if key == _key:
                return True
        return False

    def remove(self, key):
        index = self._index(key)
        bucket = self.buckets[index]
        for i, (_key, _value) in enumerate(bucket):
            if _key == key:
                bucket.pop(i)
                self.size -= 1
                return True
        return False

    def stats(self):
        lengths = [len(bucket) for bucket in self.buckets]
        return {
            "size": self.size,
            "capacity": self.capacity,
            "load_factor": self.load_factor(),
            "empty_buckets": lengths.count(0),
            "max_bucket_len": max(lengths),
        }

    def __len__(self):
        return self.size

---
# 6. Проверяем реализацию

In [ ]:
def check(name, condition):
    print(("СУПЕР!!!!" if condition else "МЛДАААА"), name)

m = MyHashMap(4)

m.put("a", 10)
check("put/get", m.get("a") == 10)

m.put("a", 20)
check("обновление существующего ключа", m.get("a") == 20)
check("size после update не вырос", len(m) == 1)

check("contains", m.contains("a"))
check("contains missing", not m.contains("missing"))

check("remove", m.remove("a") is True)
check("ключ удалён", not m.contains("a"))
check("повторный remove", m.remove("a") is False)

m = MyHashMap(4)
m.put("a", 1)
m.put("b", 2)
m.put("c", 3)          

capacity_before = m.capacity
m.put("a", 100)       

check("update не вызывает resize", m.capacity == capacity_before)
check("значение после update изменилось", m.get("a") == 100)


## Проверка коллизий

Чтобы `resize` не мешал эксперименту, возьмём `capacity = 8`

Ключи:

```text
1, 9, 17, 25
```

при `capacity = 8` имеют одинаковый остаток при делении на 8 и попадают в один bucket

Все значения всё равно должны корректно находиться


In [ ]:
m = MyHashMap(8)

for key in [1, 9, 17, 25]:
    m.put(key, key * 10)

check("collision 1", m.get(1) == 10)
check("collision 9", m.get(9) == 90)
check("collision 17", m.get(17) == 170)
check("collision 25", m.get(25) == 250)
check("resize ещё не нужен", m.capacity == 8)

m.remove(9)

check("9 удалён", m.get(9) is None)
check("1 остался", m.get(1) == 10)
check("17 остался", m.get(17) == 170)
check("25 остался", m.get(25) == 250)


## Проверка `resize` и `rehash`

При `capacity = 4`:

```text
3 / 4 = 0.75
```

Три элемента еще помещаются без расширения

Но добавление четвертого дало бы:

```text
4 / 4 = 1.0
```

Поэтому **перед добавлением четвёртого нового ключа** таблица должна расшириться:

```text
4 → 8
```

После `resize` все старые элементы должны по-прежнему находиться


In [ ]:
m = MyHashMap(4)

m.put(1, "A")
m.put(2, "B")
m.put(3, "C")

check("после 3 элементов capacity = 4", m.capacity == 4)
check("load factor = 0.75", abs(m.load_factor() - 0.75) < 1e-9)

m.put(4, "D")

check("при добавлении 4-го элемента capacity стала 8", m.capacity == 8)
check("1 не потерялся после rehash", m.get(1) == "A")
check("2 не потерялся после rehash", m.get(2) == "B")
check("3 не потерялся после rehash", m.get(3) == "C")
check("4 добавился", m.get(4) == "D")
check("load factor снова <= 0.75", m.load_factor() <= MyHashMap.MAX_LOAD_FACTOR)


---
# 7. Строим индекс на своем датасете

Теперь используем собственную Hash Table как реальный индекс:

```text
уникальный ключ → row_id
```

In [ ]:
def build_hash_index(values):
    index = MyHashMap()

    for row_id, key in enumerate(values):
        index.put(key, row_id)

    return index


In [ ]:
hash_index = build_hash_index(keys)

print("Индекс построен")
print("Количество ключей:", len(hash_index))
print("Итоговая capacity:", hash_index.capacity)
print(f"Load factor: {hash_index.load_factor():.4f}")


In [ ]:
TARGET = keys[len(keys) // 2]

row_id = hash_index.get(TARGET)

print("TARGET:", TARGET)
print("row_id:", row_id)

if row_id is not None:
    display(df.iloc[[row_id]])

---
# 8. Сравниваем два способа поиска

In [ ]:
test_positions = np.linspace(
    0,
    len(keys) - 1,
    num=min(30, len(keys)),
    dtype=int
)

test_keys = [keys[i] for i in test_positions]

start = perf_counter()
for key in test_keys:
    linear_search(keys, key)
linear_ms = (perf_counter() - start) * 1000

start = perf_counter()
for key in test_keys:
    hash_index.get(key)
hash_ms = (perf_counter() - start) * 1000

print(f"Запросов: {len(test_keys)}")
print(f"Linear search: {linear_ms:.3f} ms")
print(f"Hash lookup:   {hash_ms:.3f} ms")

if hash_ms > 0:
    print(f"Отношение времени: {linear_ms / hash_ms:.2f}x")

## Стоимость построения индекса

In [ ]:
start = perf_counter()
tmp_index = build_hash_index(keys)
build_ms = (perf_counter() - start) * 1000

print(f"Build hash index: {build_ms:.3f} ms")

# 9. Итог сегодняшней версии проекта

Заполните:

**Датасет:**  
Chicago Taxi Trips 

**Размер:**  
200 000 строк

**Уникальное поле:**  
trip_id

**Что хранит индекс:**  

```text
trip_id → row_id
```

**Поиск без индекса:**  
Линейный поиск: проходим по всем ключам подряд и сравниваем с искомым

**Сложность поиска без индекса:**  
O(N)

**Average lookup в Hash Table:**  
O(1)

**Worst-case lookup:**  
O(N)

**Почему worst-case может ухудшиться:**  
Если много ключей попадают в один bucket (плохая хеш-функция, много коллизий), то этот bucket превращается в длинный список, и поиск в нём такой же, как линейный.

**Начальная capacity:**  
8

**Итоговая capacity после загрузки датасета:**  
524288

**Итоговый load factor:**  
0.3815

**Когда происходит resize:**  
Когда добавляется новый ключ и (size + 1) / capacity получается больше 0.75. Тогда capacity увеличивается в 2 раза.

**Почему при resize нужен rehash:**  
Номер bucket считается как hash(key) % capacity. Когда capacity меняется, номер bucket у ключа тоже может поменяться, поэтому все пары надо разложить заново.

**Сколько заняло построение индекса:**  
1439.334 ms

**Сколько занял linear search:**  
391.553 ms

**Сколько занял hash lookup:**  
0.228 ms

**Какую цену мы заплатили за быстрый поиск:**  
Время на построение индекса, дополнительная память под buckets и пары (ключ, row_id), а также время на resize и rehash при росте таблицы.

---
# 10. Что будет дальше

Сегодня проект умеет только одно:

```text
быстро найти запись по уникальному ключу
```

На следующих занятиях будут появляться другие типы задач и новые структуры

Мы будем **добавлять их только тогда, когда появится конкретная проблема, которую текущая архитектура решает плохо**


---
# Дополнительно, если все основное готово

Добавьте метод:

```python
stats()
```

который возвращает или печатает:

```text
size
capacity
load_factor
количество пустых buckets
размер самого длинного bucket
```

После этого посмотрите статистику для индекса вашего датасета и коротко ответьте:

> Насколько равномерно элементы распределились по buckets?


In [ ]:
print(hash_index.stats())

Элементы распределились довольно равномерно: самый длинный bucket очень короткий по сравнению с общим числом ключей (см. вывод выше), поэтому поиск почти всегда смотрит в небольшой список.